## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting? 
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** [Sissi Wang]

**Date:** [10/02/2026]

In [3]:
# Your Phase 1 solution to the problem goes here.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score

# 1. Load the data. Perform some EDA, summarizing the target label and the features.
df = pd.read_csv('data/land_mines.csv')
print(df.describe())
print(df['mine_type'].value_counts().sort_index())

# 2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, 
# in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
X_raw = df[['voltage', 'height', 'soil']]
y = df['mine_type']

# 50/50 split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.50, random_state=42, stratify=y
)

# Feature Scaling using MinMaxScaler
feature_scaler = MinMaxScaler()

X_train_scaled = pd.DataFrame(
    feature_scaler.fit_transform(X_train_raw),
    columns=X_train_raw.columns,
    index=X_train_raw.index
)

X_test_scaled = pd.DataFrame(
    feature_scaler.transform(X_test_raw),
    columns=X_test_raw.columns,
    index=X_test_raw.index
)
# 3. Build a $k$-NN classifier. Explain how you select $k$.
eval_ks = np.arange(1, 51)
eval_valid_sse = []
eval_train_sse = []
for eval_k in eval_ks:
    eval_candidate = KNeighborsClassifier(n_neighbors=int(eval_k))
    eval_candidate.fit(X_train_scaled, y_train)
    eval_valid_hat = eval_candidate.predict(X_test_scaled)
    eval_train_hat = eval_candidate.predict(X_train_scaled)
    eval_valid_sse.append(np.sum(
        (y_test.to_numpy() - eval_valid_hat) ** 2
    ))
    eval_train_sse.append(np.sum(
        (y_train.to_numpy() - eval_train_hat) ** 2
    ))
eval_k_star = int(eval_ks[np.argmin(eval_valid_sse)])
print("Best k:", eval_k_star)
print("Best valid SSE:", min(eval_valid_sse))
# I selected the optimal k value 2 based on the minimum valid SSE.

# Optimal model
optimal_knn = KNeighborsClassifier(n_neighbors=eval_k_star)
optimal_knn.fit(X_train_scaled, y_train)
y_pred = optimal_knn.predict(X_test_scaled)
#4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. 
# How accurate is it? Where is performance more or less accurate?
print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_test, y_pred)
print(cm)

print("\n--- Accuracy ---")
print(accuracy_score(y_test, y_pred))
# The model is more accurat on predicting type 2 mine, but exihibit poor performance on type 3, 4 and 5, often mistakening them from each
# other or misclassifying them as type 1.

# 5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how
# you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.
#This optimal model achieves an overall test accuracy of 45.56%, but its performance varies drastically across landmine types. 
# It is able to correctly identify Type 2 mines with a 91.4% accuracy, it struggles significantly with other types, 
# especially Type 5, identifying only 1 out of 32 actual Type 5 mines. Even though it is able to classifiy a high number of type 1 mine,
# it misclassified more than it correctly identified. When using this model in real life, always human verify the prediction, be aware that 
# the prediction could be wrong, but have more trust in type 2 prediction


          voltage      height        soil   mine_type
count  338.000000  338.000000  338.000000  338.000000
mean     0.430634    0.508876    0.503550    2.952663
std      0.195819    0.306043    0.344244    1.419703
min      0.197734    0.000000    0.000000    1.000000
25%      0.309737    0.272727    0.200000    2.000000
50%      0.359516    0.545455    0.600000    3.000000
75%      0.482628    0.727273    0.800000    4.000000
max      0.999999    1.000000    1.000000    5.000000
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64
Best k: 2
Best valid SSE: 497

--- Confusion Matrix ---
[[26  0  6  3  1]
 [ 0 32  0  3  0]
 [12  1 10  6  4]
 [ 9  6 10  8  0]
 [11  0 12  8  1]]

--- Accuracy ---
0.4556213017751479


**Q1.**
1. What is the difference between regression and classification?
Regression predicts numerical target values, and the result will be numerical values. Classification predicts categorical values, result will be finite number of classification labels.
2. What is a confusion table? What does it help us understand about a model's performance?
It counts predictions by actual class and predicted class. It shows information on how well each model performs in individual classes instead of just a general overall accuracy
3. What does the SSE quantify about a particular model?
It measures the cumulative squared deviation between actual observed values and model predictions. Lower SSE values shows a tighter fit to the target values, where higher SSE values indicate larger overall prediction errors.
4. What are overfitting and underfitting? 
Overfitting occurs when a model is overly complex and memorizes noise, outliers, or sampling fluctuations in the training dataset. It achieves low error on training data but generalizes poorly to unseen test data. Underfitting occurs when a model is too simple or heavily constrained to capture the underlying structure or patterns in the data. It has poor performance on both training data and unseen test data.
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
Splitting data into training and test sets and evaluating hyperparameters like $k$ on unseen test data improves generalization performance by preventing data leakage and optimism bias. Splitting the data and the percentage of splitting affects the amount of data used for trainig and testing. Sometimes different way of splitting can cause bias. For different types of data, different k values has variaous performance. Choosing the best k for diff dataset can optimize overall performance or precision.
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.
Prediction approach simplifies decision making and and requires no post processing. However, it conceals model confidence (a 51% probability prediction looks identical to a 99% probability prediction).
Probablity distribution over class labels reflects confidence levels for each class and allows tuning cutoffs based on actual misclassification risks. However, it needs secondary rules or cutoffs to produce discrete outputs.

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [581858da4af2c1b4d15b1794bdd463f4efaec9bd]
- **AI tool and model used:** [Gemini 3.6 Flash]

### *Prompts used

1. [[Datasheet and notebook attached]The assignment directions is in the notebook, I have completed the steps for phase 1, please give bullet point suggestions to make my code better or better completing the assignment]
2. [Paste follow-up prompt]

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. You are calculating Sum of Squared Errors ($\text{SSE} = \sum (y - \hat{y})^2$) on categorical class labels (mine_type values $1, 2, 3, 4, 5$).Why it's wrong: Class numbers are nominal identifiers, not continuous magnitudes. Misclassifying a Type 1 mine as a Type 5 yields an error of $(1 - 5)^2 = 16$, whereas misclassifying a Type 1 mine as a Type 2 yields an error of $(1 - 2)^2 = 1$. This artificially penalizes certain class misclassifications far more than others simply because of their numerical index.Fix: Use Accuracy (or Zero-One Classification Loss / Misclassification Rate) to evaluate and select $k$ for classification tasks:
2. Choosing $k$ using Accuracy on your MinMaxScaler scaled data yields $k = 1$ (Accuracy = 46.15%), whereas SSE picks $k = 2$ because of the squared-distance artifact on class labels.
3. Precision vs. Recall: In your Q2 comment, you write "It is able to correctly identify Type 2 mines with a 91.4% accuracy..."Refinement: Standardize your terminology. $\frac{32}{35}$ represents the Recall (Sensitivity) for Type 2 mines.

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept|Didn't realize this before, but classification if we use SSE if misclassify type 1 as other types shouldn't make a difference, so for the purpose of this project and data accuracy works better.|
| 2 | Accept|Agree, as it is a resulting change from suggestion 1 too.|
| 3 | Accept|Agree as didn't know the difference and correct wording before.|

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score

# 1. Load the data. Perform some EDA, summarizing the target label and the features.
df = pd.read_csv('data/land_mines.csv')
print(df.describe())
print(df['mine_type'].value_counts().sort_index())

# 2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, 
# in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
X_raw = df[['voltage', 'height', 'soil']]
y = df['mine_type']

# 50/50 split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.50, random_state=42, stratify=y
)

# Feature Scaling using MinMaxScaler
feature_scaler = MinMaxScaler()

X_train_scaled = pd.DataFrame(
    feature_scaler.fit_transform(X_train_raw),
    columns=X_train_raw.columns,
    index=X_train_raw.index
)

X_test_scaled = pd.DataFrame(
    feature_scaler.transform(X_test_raw),
    columns=X_test_raw.columns,
    index=X_test_raw.index
)
# 3. Build a $k$-NN classifier. Explain how you select $k$.
eval_ks = np.arange(1, 51)
eval_valid_acc = []
eval_train_acc = []

for eval_k in eval_ks:
    eval_candidate = KNeighborsClassifier(n_neighbors=int(eval_k))
    eval_candidate.fit(X_train_scaled, y_train)
    
    eval_valid_hat = eval_candidate.predict(X_test_scaled)
    eval_train_hat = eval_candidate.predict(X_train_scaled)
    
    # Measure accuracy (correct classification metric)
    eval_valid_acc.append(accuracy_score(y_test, eval_valid_hat))
    eval_train_acc.append(accuracy_score(y_train, eval_train_hat))

# Optimal k based on maximum validation accuracy
eval_k_star = int(eval_ks[np.argmax(eval_valid_acc)])
print(f"\nOptimal k: {eval_k_star}")
print(f"Optimal Test Accuracy: {max(eval_valid_acc):.4f}")
# I selected the optimal k value 1 based on the maximum valid accuracy.

# Optimal model
optimal_knn = KNeighborsClassifier(n_neighbors=eval_k_star)
optimal_knn.fit(X_train_scaled, y_train)
y_pred = optimal_knn.predict(X_test_scaled)
#4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. 
# How accurate is it? Where is performance more or less accurate?
print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_test, y_pred)
print(cm)

print("\n--- Accuracy ---")
print(accuracy_score(y_test, y_pred))
# The model is more accurat on predicting type 2 mine, but exihibit poor performance on type 3, 4 and 5, often mistakening them from each
# other or misclassifying them as type 1.

# 5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how
# you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.
#This optimal model achieves an overall test accuracy of 45.56%, but its performance varies drastically across landmine types. 
# It is able to correctly identify Type 2 mines with a 91.4% recall, it struggles significantly with other types, 
# especially Type 5, identifying only 1 out of 32 actual Type 5 mines. Even though it is able to classifiy a high number of type 1 mine,
# it misclassified more than it correctly identified. When using this model in real life, always human verify the prediction, be aware that 
# the prediction could be wrong, but have more trust in type 2 prediction


          voltage      height        soil   mine_type
count  338.000000  338.000000  338.000000  338.000000
mean     0.430634    0.508876    0.503550    2.952663
std      0.195819    0.306043    0.344244    1.419703
min      0.197734    0.000000    0.000000    1.000000
25%      0.309737    0.272727    0.200000    2.000000
50%      0.359516    0.545455    0.600000    3.000000
75%      0.482628    0.727273    0.800000    4.000000
max      0.999999    1.000000    1.000000    5.000000
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64

Optimal k: 1
Optimal Test Accuracy: 0.4615

--- Confusion Matrix ---
[[21  0  4  4  7]
 [ 0 32  0  3  0]
 [ 8  0  7 10  8]
 [ 7  5  4 11  6]
 [ 7  0  9  9  7]]

--- Accuracy ---
0.46153846153846156


**Q1.**
1. What is the difference between regression and classification?
Regression predicts numerical target values, and the result will be numerical values. Classification predicts categorical values, result will be finite number of classification labels.
2. What is a confusion table? What does it help us understand about a model's performance?
It counts predictions by actual class and predicted class. It shows information on how well each model performs in individual classes instead of just a general overall accuracy
3. What does the SSE quantify about a particular model?
It measures the cumulative squared deviation between actual observed values and model predictions. Lower SSE values shows a tighter fit to the target values, where higher SSE values indicate larger overall prediction errors.
4. What are overfitting and underfitting? 
Overfitting occurs when a model is overly complex and memorizes noise, outliers, or sampling fluctuations in the training dataset. It achieves low error on training data but generalizes poorly to unseen test data. Underfitting occurs when a model is too simple or heavily constrained to capture the underlying structure or patterns in the data. It has poor performance on both training data and unseen test data.
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
Splitting data into training and test sets and evaluating hyperparameters like $k$ on unseen test data improves generalization performance by preventing data leakage and optimism bias. Splitting the data and the percentage of splitting affects the amount of data used for trainig and testing. Sometimes different way of splitting can cause bias. For different types of data, different k values has variaous performance. Choosing the best k for diff dataset can optimize overall performance or precision.
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.
Prediction approach simplifies decision making and and requires no post processing. However, it conceals model confidence (a 51% probability prediction looks identical to a 99% probability prediction).
Probablity distribution over class labels reflects confidence levels for each class and allows tuning cutoffs based on actual misclassification risks. However, it needs secondary rules or cutoffs to produce discrete outputs.

### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]

The use of artificial intelligence pointed out minor flaws in my code and overall made it more accurate and efficient with explanations. In phase 1, I was able to successfully solve the problems stated, but improvements could have been made. These changes are reflected in my phase 2 of the assignment. For this assignment, I found AI to be a lot more beneficial to me, as the suggestion it gave actually improved my accuracy in the overall model. In the past assignments, AI assisted in streamlining my code and improving flow, but in this assignment it made visible recognizable improvements to the AI model. When it came to evaluating the changes that AI suggested, I accepted the tasks because it directly made my code better in terms of accuracy of the model. AI is not necessarily limited in this case, as it pointed out the loophole in my understanding and therefore made my model better.